# DQ 检查 — ODS 层（Rule 0a/0b）

在 DWD 之前拦截两类问题：
- **Rule 0a**：每个来源的 ODS 分区行数 > 0（空文件/文件未到检测）
- **Rule 0b**：各月行数与其他月均值偏差 < 50%（异常波动检测）

任何 FAIL 将抛出异常，阻断 DWD 构建。

## 1. SparkSession

In [1]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-DQ-AfterODS")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE    = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y   = f"{BASE}/data/warehouse/ods/yellow"
ODS_G   = f"{BASE}/data/warehouse/ods/green"
ODS_FHV = f"{BASE}/data/warehouse/ods/fhv_hv"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready | TARGET_DATE={TARGET_DATE}")


26/10/19 14:55:35 WARN Utils: Your hostname, jelly-macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.229 instead (on interface en0)
26/10/19 14:55:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 14:55:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TARGET_DATE=2024-03-15
Spark 3.5.1 ready | TARGET_DATE=2024-03-15


## 2. DQ 框架

In [2]:
results = []

def check(rule_id, layer, rule_type, description, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"rule_id": rule_id, "status": status, "description": description})
    mark = "✅" if passed else "❌"
    print(f"{mark} Rule {rule_id} [{layer}] {description}  {detail}")


## 3. Rule 0a：各来源分区行数非空

In [ ]:
from datetime import date, timedelta

sources = [
    ("yellow", ODS_Y),
    ("green",  ODS_G),
    ("fhv_hv", ODS_FHV),
]

target_dt = date(yr, mo, day)
# Previous 7 days for rolling average baseline
prev_7_dates = [(target_dt - timedelta(days=i)) for i in range(1, 8)]

for src, path in sources:
    # Read ALL available ODS days for this source (no single-day filter)
    df_all = spark.read.parquet(path)
    daily_counts = {
        (r.pickup_year, r.pickup_month, r.pickup_day): r["count"]
        for r in df_all.groupBy("pickup_year", "pickup_month", "pickup_day")
                       .count().collect()
    }

    today_count = daily_counts.get((yr, mo, day), 0)

    # Rule 0a: target partition must be non-empty
    check(f"0a-{src}", "ODS", "non_empty",
          f"{src} partition non-empty for {TARGET_DATE}",
          passed=(today_count > 0),
          detail=f"today={today_count:,}  total_days_in_ODS={len(daily_counts)}")

    # Rule 0b: today's count vs rolling 7-day average ≤30% deviation
    # Requires ≥2 days of prior history; skipped gracefully on first run
    past_counts = [
        daily_counts[(d.year, d.month, d.day)]
        for d in prev_7_dates
        if (d.year, d.month, d.day) in daily_counts
    ]

    if len(past_counts) >= 2:
        avg_past = sum(past_counts) / len(past_counts)
        deviation = abs(today_count - avg_past) / avg_past if avg_past > 0 else 0
        check(f"0b-{src}", "ODS", "row_volatility",
              f"{src} daily count vs {len(past_counts)}-day rolling avg ≤30%",
              passed=(deviation < 0.30),
              detail=f"today={today_count:,}  {len(past_counts)}-day_avg={avg_past:,.0f}  deviation={deviation:.1%}")
    else:
        print(f"⚠️  {src}: only {len(past_counts)} prior day(s) in ODS — volatility check requires ≥2, skipped")


## 4. Rule 0b：月度行数波动检测（偏差 < 50%）

In [ ]:
# Rule 0a and 0b are now fully handled in the cell above (c07).
# Each source's daily count is checked against the rolling 7-day average in one loop.
# This cell is intentionally left as a no-op to preserve notebook cell numbering.
print("Rules 0a/0b checks complete (handled in previous cell).")


## 5. 报告 + 阻断判断

In [5]:
fail_rules = [r for r in results if r['status'] == 'FAIL']
pass_rules = [r for r in results if r['status'] == 'PASS']

print(f"\n{'='*50}")
print(f"DQ Report (ODS)  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Rules: {len(results)} total | {len(pass_rules)} PASS | {len(fail_rules)} FAIL")
print(f"={'='*49}")

spark.stop()

if fail_rules:
    for r in fail_rules:
        print(f"  ❌ {r['rule_id']}: {r['description']}")
    raise ValueError(
        f"DQ FAILED: {len(fail_rules)} ODS rule(s). "
        "Pipeline halted — check source data before DWD build."
    )
else:
    print("All ODS rules passed — proceeding to DWD.")

print("DQ after-ODS check complete.")



DQ Report (ODS)  2026-10-19 14:55
Rules: 3 total | 3 PASS | 0 FAIL


All ODS rules passed — proceeding to DWD.
DQ after-ODS check complete.
